# Классификация цвета автомобиля

Классификация цвета автомобиля на датасете **DVM** (используются фронтальные изображения).

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import time
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.io import read_image
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

In [ ]:
def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def fmt_seconds(s: float) -> str:
    s = int(round(s))
    h, s = divmod(s, 3600)
    m, s = divmod(s, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

In [ ]:
import kagglehub

@dataclass
class CFG:
    data_dir: Path = Path(kagglehub.dataset_download("juliancotto/car-fronts-image-dataset-from-dvm-car-dataset"))
    seed: int = 555
    img_size: int = 224
    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 20
    lr_scratch: float = 1e-2
    lr_ft: float = 3e-4
    weight_decay: float = 1e-4
    patience: int = 3
    min_class_count: int = 50


cfg = CFG()

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(cfg.seed)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='mps')

In [ ]:
def index_from_folders(data_dir: Path) -> pd.DataFrame:
    rows = []
    for p in data_dir.rglob("*"):
        if not (p.is_file() and p.suffix.lower() == ".jpg"):
            continue

        name = p.name
        colour = None

        if "$$" in name:
            toks = name.split("$$")
            if len(toks) >= 4:
                colour = toks[3]

        if colour is None:
            continue

        rows.append({"path": str(p), "colour": str(colour)})

    return pd.DataFrame(rows)


df = index_from_folders(cfg.data_dir)

df["colour"] = df["colour"].astype(str).str.strip().str.lower()

# filter for correct stratify and stable F1_macro
vc = df["colour"].value_counts()
keep = vc[vc >= cfg.min_class_count].index
df = df[df["colour"].isin(keep)].reset_index(drop=True)

df.head(), df["colour"].nunique(), len(df)

(                                                path colour
 0  confirmed_fronts/SEAT/2013/SEAT$$Altea$$2013$$...   grey
 1  confirmed_fronts/SEAT/2013/SEAT$$Alhambra$$201...   blue
 2  confirmed_fronts/SEAT/2013/SEAT$$Mii$$2013$$Re...    red
 3  confirmed_fronts/SEAT/2013/SEAT$$Exeo$$2013$$G...   grey
 4  confirmed_fronts/SEAT/2013/SEAT$$Mii$$2013$$Wh...  white,
 17,
 61755)

In [ ]:
df["colour"].value_counts().head(20)

colour
black          14317
grey            9474
white           9395
blue            8483
silver          7770
red             6095
unlisted        1516
brown            911
green            777
yellow           667
beige            600
orange           559
purple           362
bronze           329
gold             217
multicolour      196
pink              87
Name: count, dtype: int64

In [ ]:
y = df["colour"].values
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.1, random_state=cfg.seed, stratify=y
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.1111111111, random_state=cfg.seed, stratify=y[train_idx]
)  # 0.1 / 0.9 = 0.111.., for: 80/10/10

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

df_train["colour"].value_counts().head()

colour
black     11453
grey       7580
white      7516
blue       6787
silver     6216
Name: count, dtype: int64

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(int(cfg.img_size * 1.14)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class DVMColorDataset(Dataset):
    def __init__(self, df: pd.DataFrame, classes: list[str], tfms):
        self.paths = df["path"].tolist()
        self.y = df["colour"].tolist()
        self.cls2id = {c: i for i, c in enumerate(classes)}
        self.tfms = tfms

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        x = read_image(self.paths[i])  # uint8, [C,H,W]
        x = self.tfms(x)
        y = self.cls2id[self.y[i]]
        return x, y


classes = sorted(df_train["colour"].unique().tolist())
num_classes = len(classes)

train_ds = DVMColorDataset(df_train, classes, train_tfms)
val_ds = DVMColorDataset(df_val, classes, eval_tfms)
test_ds = DVMColorDataset(df_test, classes, eval_tfms)

pin = (device.type == "cuda")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=pin, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=pin)

(17, torch.Size([64, 3, 224, 224]))

## Модель №1: ResNet-18

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(identity)
        out = F.relu(out + identity, inplace=True)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes):
        super().__init__()
        self.in_ch = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_ch, blocks, stride):
        layers = [block(self.in_ch, out_ch, stride)]
        self.in_ch = out_ch * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        x = self.fc(x)
        return x


def resnet18_scratch(num_classes: int) -> nn.Module:
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

## Модели №2-3: pretrained + fine-tuning

2 предобученные модели:
- **ResNet-50 (ImageNet)**  
- **MobileNetV3-Large (ImageNet)**

In [ ]:
def build_resnet50_pretrained(num_classes: int) -> nn.Module:
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def build_mobilenetv3_pretrained(num_classes: int) -> nn.Module:
    m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    return m

In [ ]:
@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x).detach().cpu()
        all_logits.append(logits)
        all_y.append(y)
    return torch.cat(all_logits).numpy(), torch.cat(all_y).numpy()


@torch.no_grad()
def f1_macro(model, loader) -> float:
    logits, y = predict_logits(model, loader)
    pred = logits.argmax(axis=1)
    return float(f1_score(y, pred, average="macro"))


def train_one_epoch(model, loader, opt, scaler, loss_fn, epoch: int, epochs: int):
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc=f"train {epoch}/{epochs}", leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            logits = model(x)
            loss = loss_fn(logits, y)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        bs = x.size(0)
        total_loss += loss.detach().item() * bs
        pbar.set_postfix(loss=float(loss.detach().item()))

    return total_loss / len(loader.dataset)


def fit(model, train_loader, val_loader, epochs, lr, weight_decay, patience):
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_f1 = -1.0
    best_state = None
    bad = 0

    fit_start = time.perf_counter()
    for ep in range(1, epochs + 1):
        sync_device(device)
        t0 = time.perf_counter()

        tr_loss = train_one_epoch(model, train_loader, opt, scaler, loss_fn, ep, epochs)
        sched.step()
        val_f1 = f1_macro(model, val_loader)

        sync_device(device)
        epoch_time = time.perf_counter() - t0

        elapsed = time.perf_counter() - fit_start
        progress = ep / epochs
        eta = elapsed / progress - elapsed

        print(
            f"epoch {ep:02d}/{epochs} | "
            f"train_loss={tr_loss:.4f} | val_f1_macro={val_f1:.4f} | "
            f"epoch={fmt_seconds(epoch_time)} | eta={fmt_seconds(eta)}"
        )

        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("early stop")
                break

    model.load_state_dict(best_state)
    total_time = time.perf_counter() - fit_start
    print("fit_time:", fmt_seconds(total_time))
    return model, best_f1

In [ ]:
CKPT_DIR = Path("./checkpoints")
CKPT_DIR.mkdir(exist_ok=True)


def save_ckpt(path: Path, model: nn.Module, best_val_f1: float):
    torch.save({
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "classes": classes,
        "num_classes": num_classes,
        "img_size": cfg.img_size,
        "best_val_f1": float(best_val_f1),
    }, path)


def load_ckpt(path: Path, model: nn.Module):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    return ckpt

In [ ]:
results = []

In [ ]:
# Scratch ResNet-18
ckpt_path = CKPT_DIR / "ResNet18_scratch.pth"
m1 = resnet18_scratch(num_classes)
if ckpt_path.exists():
    ckpt = load_ckpt(ckpt_path, m1)
    best_val_f1 = ckpt.get("best_val_f1", None)
    if best_val_f1 is None:
        m1 = m1.to(device)
        best_val_f1 = f1_macro(m1, val_loader)
    else:
        best_val_f1 = float(best_val_f1)
else:
    m1, best_val_f1 = fit(m1, train_loader, val_loader,
                          epochs=cfg.epochs, lr=cfg.lr_scratch,
                          weight_decay=cfg.weight_decay, patience=cfg.patience)
    save_ckpt(ckpt_path, m1, best_val_f1)

m1 = m1.to(device)
test_f1 = f1_macro(m1, test_loader)
results.append({"model": "ResNet18_scratch", "val_f1_macro": best_val_f1, "test_f1_macro": test_f1})

In [ ]:
# Pretrained ResNet-50
ckpt_path = CKPT_DIR / "ResNet50_pretrained_ft.pth"
m2 = build_resnet50_pretrained(num_classes)
if ckpt_path.exists():
    ckpt = load_ckpt(ckpt_path, m2)
    best_val_f1 = ckpt.get("best_val_f1", None)
    if best_val_f1 is None:
        m2 = m2.to(device)
        best_val_f1 = f1_macro(m2, val_loader)
    else:
        best_val_f1 = float(best_val_f1)
else:
    m2, best_val_f1 = fit(m2, train_loader, val_loader,
                          epochs=cfg.epochs, lr=cfg.lr_ft,
                          weight_decay=cfg.weight_decay, patience=cfg.patience)
    save_ckpt(ckpt_path, m2, best_val_f1)

m2 = m2.to(device)
test_f1 = f1_macro(m2, test_loader)
results.append({"model": "ResNet50_pretrained_ft", "val_f1_macro": best_val_f1, "test_f1_macro": test_f1})

In [ ]:
# Pretrained MobileNetV3-Large
ckpt_path = CKPT_DIR / "MobileNetV3_pretrained_ft.pth"
m3 = build_mobilenetv3_pretrained(num_classes)
if ckpt_path.exists():
    ckpt = load_ckpt(ckpt_path, m3)
    best_val_f1 = ckpt.get("best_val_f1", None)
    if best_val_f1 is None:
        m3 = m3.to(device)
        best_val_f1 = f1_macro(m3, val_loader)
    else:
        best_val_f1 = float(best_val_f1)
else:
    m3, best_val_f1 = fit(m3, train_loader, val_loader,
                          epochs=cfg.epochs, lr=cfg.lr_ft,
                          weight_decay=cfg.weight_decay, patience=cfg.patience)
    save_ckpt(ckpt_path, m3, best_val_f1)

m3 = m3.to(device)
test_f1 = f1_macro(m3, test_loader)
results.append({"model": "MobileNetV3_pretrained_ft", "val_f1_macro": best_val_f1, "test_f1_macro": test_f1})

In [ ]:
res_df = pd.DataFrame(results).sort_values("test_f1_macro", ascending=False)
res_df

,model,val_f1_macro,test_f1_macro
2,MobileNetV3_pretrained_ft,0.773830,0.830300
1,ResNet50_pretrained_ft,0.774929,0.824814
0,ResNet18_scratch,0.660108,0.647797


In [ ]:
# Отчёт по лучшей модели (test)
best_name = res_df.iloc[0]["model"]
best_model = {"ResNet18_scratch": m1, "ResNet50_pretrained_ft": m2, "MobileNetV3_pretrained_ft": m3}[best_name]

logits, y = predict_logits(best_model, test_loader)
pred = logits.argmax(axis=1)

print("Best model:", best_name)
print("Test F1_macro:", f1_score(y, pred, average="macro"))
print()
print(classification_report(y, pred, target_names=classes, digits=4, zero_division=0))

Best model: MobileNetV3_pretrained_ft
Test F1_macro: 0.8303000296955999

              precision    recall  f1-score   support

       beige     0.9661    0.9500    0.9580        60
       black     0.9854    0.9867    0.9860      1432
        blue     0.9767    0.9870    0.9818       848
      bronze     0.9655    0.8485    0.9032        33
       brown     0.9000    0.9890    0.9424        91
        gold     0.9000    0.8182    0.8571        22
       green     0.0000    0.0000    0.0000        78
        grey     0.0000    0.0000    0.0000       947
 multicolour     0.8500    0.8500    0.8500        20
      orange     0.9310    0.9643    0.9474        56
        pink     1.0000    1.0000    1.0000         9
      purple     1.0000    0.8889    0.9412        36
         red     0.9885    0.9852    0.9868       609
      silver     0.9795    0.9820    0.9807       777
    unlisted     0.9015    0.7829    0.8380       152
       white     0.9841    0.9915    0.9878       939
      ye

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, pred)

cls2id = {c: i for i, c in enumerate(classes)}

for c in ["green", "grey"]:
    if c in cls2id:
        i = cls2id[c]
        row = cm[i]
        top = np.argsort(row)[::-1][:10]
        print(f"\nTrue class = {c} (id={i}), support={row.sum()}")
        for j in top:
            if row[j] > 0:
                print(f"  predicted as {classes[j]}: {row[j]}")


True class = green (id=6), support=78
  predicted as grey: 75
  predicted as beige: 1
  predicted as brown: 1
  predicted as white: 1

True class = grey (id=7), support=947
  predicted as green: 921
  predicted as black: 9
  predicted as silver: 7
  predicted as unlisted: 4
  predicted as white: 3
  predicted as blue: 1
  predicted as brown: 1
  predicted as gold: 1


По confusion matrix обнаружена полная взаимная подмена классов green и grey. Если вручную поменять местами классы, F1 окажется сильно лучше:

In [ ]:
pred_fixed = pred.copy()

cls2id = {c: i for i, c in enumerate(classes)}
if "green" in cls2id and "grey" in cls2id:
    g = cls2id["green"]
    gy = cls2id["grey"]
    pred_fixed[pred == g] = -1
    pred_fixed[pred == gy] = g
    pred_fixed[pred_fixed == -1] = gy

print("New F1_macro:", f1_score(y, pred_fixed, average="macro"))
print(classification_report(y, pred_fixed, target_names=classes, digits=4, zero_division=0))

New F1_macro: 0.9446235390036293
              precision    recall  f1-score   support

       beige     0.9661    0.9500    0.9580        60
       black     0.9854    0.9867    0.9860      1432
        blue     0.9767    0.9870    0.9818       848
      bronze     0.9655    0.8485    0.9032        33
       brown     0.9000    0.9890    0.9424        91
        gold     0.9000    0.8182    0.8571        22
       green     0.9868    0.9615    0.9740        78
        grey     0.9664    0.9725    0.9695       947
 multicolour     0.8500    0.8500    0.8500        20
      orange     0.9310    0.9643    0.9474        56
        pink     1.0000    1.0000    1.0000         9
      purple     1.0000    0.8889    0.9412        36
         red     0.9885    0.9852    0.9868       609
      silver     0.9795    0.9820    0.9807       777
    unlisted     0.9015    0.7829    0.8380       152
       white     0.9841    0.9915    0.9878       939
      yellow     0.9692    0.9403    0.9545     

In [ ]:
p = CKPT_DIR / "MobileNetV3_pretrained_ft.pth"

if p.exists():
    ck = torch.load(p, map_location="cpu")
    ck_classes = ck.get("classes", None)
    print("current classes:", list(enumerate(classes)))
    print("ckpt classes   :", list(enumerate(ck_classes)) if ck_classes is not None else None)
    print()
    print("classes equal? :", ck_classes == classes)

current classes: [(0, 'beige'), (1, 'black'), (2, 'blue'), (3, 'bronze'), (4, 'brown'), (5, 'gold'), (6, 'green'), (7, 'grey'), (8, 'multicolour'), (9, 'orange'), (10, 'pink'), (11, 'purple'), (12, 'red'), (13, 'silver'), (14, 'unlisted'), (15, 'white'), (16, 'yellow')]
ckpt classes   : [(0, 'beige'), (1, 'black'), (2, 'blue'), (3, 'bronze'), (4, 'brown'), (5, 'gold'), (6, 'gray'), (7, 'green'), (8, 'multicolour'), (9, 'orange'), (10, 'pink'), (11, 'purple'), (12, 'red'), (13, 'silver'), (14, 'unlisted'), (15, 'white'), (16, 'yellow')]

classes equal? : False


Проблема в 'grey' и 'gray' ('grey' < 'green' < 'gray').

### Вывод
MobileNetV3 оказалась наиболее точной с f1-macro = 0.830300, но если обучить её еще раз, с исправленным 'grey', результат должен стать ~0.95 (проверено на тестовых данных, если поменять местами с 'green'). Оценка очень близкая к ResNet50, но достаточно далека от обученной с нуля ResNet18, что ожидаемо, так как предобученные модели обучились на более крупном датасете + DVM, а ResNet18 только на DVM. Кроме того, MobileNetV3 была обучена на 20 эпохах вместо 15, как у остальных.

Более высокая точность на тестах связана с отсутствием аугментации в этой выборке.